- &#9989; compare all radiomics features across 4 parametric maps (Boxplots and K-W test)
- &#9989; table of significance

In [16]:
all_four = 0
just_bmd = 0
just_vmi40 = 0
just_casupp25 = 0
just_konv = 0
casupp_vmi40 = 0
casupp_vmi40_bmd = 0

vmi40 = 0
casupp25 = 0
konv = 0
bmd = 0

In [17]:
from helper_files.feature_selection_helper import *
sns.set_style("darkgrid")

clinical_df = pd.read_csv(r"E:\Clinical_data\Table_clinical_data.csv", encoding="cp1252")
all_csvs = merged_lesions_csv(r"E:\DATA_Myelomy")

df_vmi40 = all_csvs["monoe_40kev"]
df_casupp25 = all_csvs["CaSupp_25"]
df_bmd = all_csvs["BMD"]
df_konv = all_csvs["konv"]

numeric_radiomics_features = df_vmi40.select_dtypes(include=['number']).columns

merged_df_vmi40 = pd.merge(df_vmi40, clinical_df, left_on='patient', right_on='Patient ID')
merged_df_casupp25 = pd.merge(df_casupp25, clinical_df, left_on='patient', right_on='Patient ID')
merged_df_bmd = pd.merge(df_bmd, clinical_df, left_on='patient', right_on='Patient ID')
merged_df_konv = pd.merge(df_konv, clinical_df, left_on='patient', right_on='Patient ID')

for f in numeric_radiomics_features:

    subset_vmi40 = merged_df_vmi40[[f, "ISS classification"]].dropna()
    subset_casupp25 = merged_df_casupp25[[f, "ISS classification"]].dropna()
    subset_bmd = merged_df_bmd[[f, "ISS classification"]].dropna()
    subset_konv = merged_df_konv[[f, "ISS classification"]].dropna()

    groups_vmi40 = [group[f].values for name, group in subset_vmi40.groupby("ISS classification")]
    groups_casupp25 = [group[f].values for name, group in subset_casupp25.groupby("ISS classification")]
    groups_bmd = [group[f].values for name, group in subset_bmd.groupby("ISS classification")]
    groups_konv = [group[f].values for name, group in subset_konv.groupby("ISS classification")]

    stat_vmi40, p_value_vmi40 = kruskal(*groups_vmi40)
    stat_casupp25, p_value_casupp25 = kruskal(*groups_casupp25)
    stat_bmd, p_value_bmd = kruskal(*groups_bmd)
    stat_konv, p_value_konv = kruskal(*groups_konv)

    if p_value_vmi40 < 0.05 or p_value_casupp25 < 0.05 or p_value_bmd < 0.05 or p_value_konv < 0.05:

        all_four += int(p_value_vmi40 < 0.05 and p_value_casupp25 < 0.05 and p_value_bmd < 0.05 and p_value_konv < 0.05)

        just_bmd += int(p_value_vmi40 >= 0.05 and p_value_casupp25 >= 0.05 and p_value_bmd < 0.05 and p_value_konv >= 0.05)
        just_vmi40 += int(p_value_vmi40 < 0.05 and p_value_casupp25 >= 0.05 and p_value_bmd >= 0.05 and p_value_konv >= 0.05)
        just_casupp25 += int(p_value_vmi40 >= 0.05 and p_value_casupp25 < 0.05 and p_value_bmd >= 0.05 and p_value_konv >= 0.05)
        just_konv += int(p_value_vmi40 >= 0.05 and p_value_casupp25 >= 0.05 and p_value_bmd >= 0.05 and p_value_konv < 0.05)

        casupp_vmi40 += int(p_value_vmi40 < 0.05 and p_value_casupp25 < 0.05 and p_value_bmd >= 0.05 and p_value_konv >= 0.05)
        casupp_vmi40_bmd += int(p_value_vmi40 < 0.05 and p_value_casupp25 < 0.05 and p_value_bmd < 0.05 and p_value_konv >= 0.05)

        vmi40 += int(p_value_vmi40 < 0.05)
        casupp25 += int(p_value_casupp25 < 0.05)
        bmd += int(p_value_bmd < 0.05)
        konv += int(p_value_konv < 0.05)

        # plt.figure(figsize=(12, 10))
        #
        # plt.subplot(221)
        # sns.boxplot(data=merged_df_vmi40, x="ISS classification", y=f)
        # plt.title(f"VMI40\np: {p_value_vmi40:.4f} --- stat: {stat_vmi40:.4f} --- sig: {p_value_vmi40 < 0.05}")
        #
        # plt.subplot(222)
        # sns.boxplot(data=merged_df_casupp25, x="ISS classification", y=f)
        # plt.title(f"CaSupp 25\np: {p_value_casupp25:.4f} --- stat: {stat_casupp25:.4f} --- sig: {p_value_casupp25 < 0.05}")
        #
        # plt.subplot(223)
        # sns.boxplot(data=merged_df_bmd, x="ISS classification", y=f)
        # plt.title(f"BMD\np: {p_value_bmd:.4f} --- stat: {stat_bmd:.4f} --- sig: {p_value_bmd < 0.05}")
        #
        # plt.subplot(224)
        # sns.boxplot(data=merged_df_konv, x="ISS classification", y=f)
        # plt.title(f"Conv CT\np: {p_value_konv:.4f} --- stat: {stat_konv:.4f} --- sig: {p_value_konv < 0.05}")
        #
        # print()

        # plt.tight_layout()
        # plt.savefig(fr"E:\Graphs\{f}.png", dpi=300, bbox_inches='tight')
        # plt.show()

In [18]:
print("--------------- Output table ---------------")
print(f"all four significant:                  {all_four}")
print()
print(f"just VMI40 significant:                {just_vmi40}")
print(f"just CaSupp_25 significant:            {just_casupp25}")
print(f"just BMD significant:                  {just_bmd}")
print(f"just konv significant:                 {just_konv}")
print()
print(f"CaSupp_25 + VMI40 significant:         {casupp_vmi40}")
print(f"CaSupp_25 + VMI40 + BMD significant:   {casupp_vmi40_bmd}")
print()
print("--------------- Count table ---------------")
print(f"VMI40:          {vmi40}")
print(f"Casupp25:       {casupp25}")
print(f"BMD:            {bmd}")
print(f"Konv:           {konv}")

--------------- Output table ---------------
all four significant:                  75

just VMI40 significant:                7
just CaSupp_25 significant:            7
just BMD significant:                  1
just konv significant:                 1

CaSupp_25 + VMI40 significant:         0
CaSupp_25 + VMI40 + BMD significant:   3

--------------- Count table ---------------
VMI40:          123
Casupp25:       112
BMD:            84
Konv:           110
